### Loading the existing topology and pricing structure

In [12]:
import pricing_driven_service_allocation as pdsa
TOPOLOGIES_RESULT_DIR = "synthetic-dataset/synthetic-topologies"
DATASET_RESULT_DIR = "synthetic-dataset/data"
UNLIMITED_VALUE = 100000000
topology_id= "7f2bb738-0766-4314-b90d-838d7c8f91d4"
C_LOCATIONS_DATASET_PATH = "eua-dataset/users/users-aus.csv"

### 1. Load all client locations

In [13]:
c_locations_df = pdsa.dataset.load_c_locations_dataframe(C_LOCATIONS_DATASET_PATH)

import os
c_locations_df.to_csv(os.path.join(DATASET_RESULT_DIR, "client_locations.csv"))

print("Dataset size:", len(c_locations_df))
c_locations_df.head()

Dataset size: 4748


,latitude,longitude
0,-30.5083,151.6712
1,-37.8833,145.3333
2,-32.2430,148.6048
3,-37.8141,144.9630
4,-27.4891,153.0188


### 2. Filter area and number of clients

In [14]:
import pandas as pd
from shapely.geometry import Point, Polygon

c_locations_df = pd.read_csv(os.path.join(DATASET_RESULT_DIR, "client_locations.csv"), index_col=0)

import yaml

# Load the YAML file
with open(os.path.join(DATASET_RESULT_DIR, "experiment_boundaries.yml"), 'r') as file:
    data = yaml.safe_load(file)

# Access experimental parameters
client_density = data['experiments']['small']['client_density']
client_area = data['experiments']['small']['box_coordinates']

# Filter only those clients that are in the polygon
poly = Polygon(client_area)
mask = c_locations_df.apply(lambda row: poly.contains(Point(row['longitude'], row['latitude'])), axis=1)
f_area_df = c_locations_df[mask]

# Filter a subset of the clients to have density from [0,1]
f_clients_df = f_area_df.sample(n=int(len(f_area_df) * client_density), random_state=35)
f_clients_df.to_csv(os.path.join(DATASET_RESULT_DIR, "client_locations_filtered.csv"), index=False)

print("Dataset size:", len(f_clients_df))
f_clients_df.head()

Dataset size: 21


,latitude,longitude
3657,-37.8146,144.970
2813,-37.8137,144.974
3342,-37.8187,144.958
4177,-37.8168,144.966
2993,-37.8151,144.959


### 3. Transform clients into resource demand

In [19]:
# TODO: Create a function that converts clients, filtered to a specific area, into a hardware demand
#  This won't be rocket science, but use maybe two parameters (application type, etc) for the transform

import numpy as np
c_locations_df = pd.read_csv(os.path.join(DATASET_RESULT_DIR, "client_locations_filtered.csv"), index_col=0)

def calculate_resources(
    n_clients: int,
    service_type: str,
    concurrency: float = 1.0,
    random_state: int = None
):
    """
    Calculates resource demand for specific Smart Environment services.

    Args:
        n_clients: Number of potential clients in the polygon.
        service_type: One of 'ar_vr', 'video_privacy', 'lidar', 'robot_iot'.
        concurrency: 0.0 to 1.0 (percent of clients active simultaneously).
        random_state: Seed for reproducibility (or None for random variation).

    Returns:
        dict: A Resource Vector (Bandwidth, Compute, Storage, Energy)
    """

    # Initialize Random Generator
    rng = np.random.default_rng(random_state)

    # 1. Define Service Profiles (The "Weights")
    # These values are examples; in production, you would calibrate these
    # based on your hardware benchmarks.
    profiles = {
        'ar_vr': {
            # High render needs, high downlink
            'bw_mbps_avg': 50,    'bw_std': 15,
            'gpu_tflops_avg': 0.5, 'gpu_std': 0.1,
            'energy_w_avg': 10,    # Energy cost at the edge node per user
            'desc': 'Ultra-low latency, render heavy'
        },
        'video_privacy': {
            # Privacy = Heavy local processing, low output bandwidth
            'bw_mbps_avg': 2,     'bw_std': 0.5, # Only metadata sent out
            'gpu_tflops_avg': 1.2, 'gpu_std': 0.2, # Heavy AI inference
            'energy_w_avg': 15,
            'desc': 'Local privacy preservation (Edge AI)'
        },
        'lidar': {
            # Massive data upload
            'bw_mbps_avg': 100,   'bw_std': 40,
            'gpu_tflops_avg': 0.8, 'gpu_std': 0.3,
            'energy_w_avg': 20,
            'desc': 'Volumetric data upload'
        },
        'robot_iot': {
            # Low individual resource, but energy critical
            'bw_mbps_avg': 0.1,   'bw_std': 0.05,
            'gpu_tflops_avg': 0.01,'gpu_std': 0.005,
            'energy_w_avg': 0.5,   # Low per unit, but high density adds up
            'desc': 'Latency sensitive, battery constraints'
        }
    }

    if service_type not in profiles:
        raise ValueError(f"Service type must be one of {list(profiles.keys())}")

    profile = profiles[service_type]

    # 2. Determine Active Users
    # We use a Binomial distribution to simulate how many are actually active
    # (More accurate than just n * concurrency)
    n_active = rng.binomial(n=n_clients, p=concurrency)

    if n_active == 0:
        return {'active_users': 0, 'bandwidth_mbps': 0, 'compute_tflops': 0, 'energy_watts': 0}

    # 3. Calculate Resources (Stochastic Simulation)
    # We simulate individual demand variations for active users

    # Bandwidth Demand
    bw_demand = rng.normal(profile['bw_mbps_avg'], profile['bw_std'], n_active)
    # Ensure no negative values (clip at 0)
    bw_total = np.maximum(bw_demand, 0).sum()

    # Compute Demand (GPU)
    gpu_demand = rng.normal(profile['gpu_tflops_avg'], profile['gpu_std'], n_active)
    gpu_total = np.maximum(gpu_demand, 0).sum()

    # Energy Estimate (Linear for simplicity, or add variance if needed)
    energy_total = n_active * profile['energy_w_avg']

    return {
        'service_mode': service_type,
        'active_users': n_active,
        'bandwidth_total_mbps': round(bw_total, 2),
        'compute_total_tflops': round(gpu_total, 2),
        'energy_total_watts': round(energy_total, 2)
    }

# --- Example Usage ---

number_clients = len(c_locations_df)

# 1. 50 People doing AR (Gaming event)
ar_reqs = calculate_resources(n_clients=number_clients, service_type='ar_vr', concurrency=0.8, random_state=42)
print("AR Requirements:", ar_reqs, "\n")

# 2. 50 Security Cameras (Privacy Mode)
vid_reqs = calculate_resources(n_clients=number_clients, service_type='video_privacy', concurrency=1.0, random_state=42)

print("Privacy Video Requirements:", vid_reqs)

AR Requirements: {'service_mode': 'ar_vr', 'active_users': 15, 'bandwidth_total_mbps': np.float64(731.97), 'compute_total_tflops': np.float64(7.85), 'energy_total_watts': 150} 

Privacy Video Requirements: {'service_mode': 'video_privacy', 'active_users': 21, 'bandwidth_total_mbps': np.float64(41.09), 'compute_total_tflops': np.float64(25.94), 'energy_total_watts': 315}


### 4. Use demand for solving the pricing

In [ ]:
import os
from iPricing.model.iPricing_pb2 import Pricing

# Convert YAML to Protocol Buffer using the package function
pricing_obj = pdsa.utils.yaml_to_pricing_proto(
    os.path.join(TOPOLOGIES_RESULT_DIR, topology_id, "pricing.yml"),
    Pricing
)

In [12]:
import json

# Custom request configuration
custom_request = {
    'currency': 'USD',
    'users_location': [
        # This limits to melbourne area
        (144.8588538568788, -37.84750988098617),
        (144.8588538568788, -37.857590074634246),
        (144.87928006127504, -37.857590074634246),
        (144.87928006127504, -37.84750988098617),
        (144.8588538568788, -37.84750988098617)
    ],
    'providers_to_consider': ['telstra'],
    'budget': 1000,
    'max_devices': 2,
    # TODO: Think about limiting the request to certain device types
    'device_types': ['CAMERA', 'SENSOR', 'DATA_CENTER'],
    'resources': {
        'available_ram': 5,
        'available_storage': 500,
        'available_vcpu': 4,
        'available_gpu': 1,
        'available_tpu': 0,
    },
    'max_distance': 10000,  # in meters
}

# Generate problem instance using the package function
problem_instance_pricing, filter_criteria = pdsa.generators.problem_instance(
    instance_pricing=pricing_obj,
    request=custom_request,
    topologies_result_dir=TOPOLOGIES_RESULT_DIR,
    unlimited_value=UNLIMITED_VALUE
)

print("\nGenerated problem instance pricing and filter criteria:")
print(f"- Original number of add-ons: {len(pricing_obj.addOns)}")
print(f"- Number of add-ons: {len(problem_instance_pricing.addOns)}")
print(f"- Filter criteria: {json.dumps(filter_criteria, indent=2)}")

Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, line 0)
Error evaluating price expression '': invalid syntax (<string>, 

In [11]:
pdsa.utils.pricing_proto_to_yaml(
    problem_instance_pricing,
    os.path.join(TOPOLOGIES_RESULT_DIR, topology_id, "problem_instance_pricing.yml")
)

Generated pricing saved to: synthetic-dataset/synthetic-topologies/7f2bb738-0766-4314-b90d-838d7c8f91d4/problem_instance_pricing.yml
